# 02 — Train baselines (scaffold)
This scaffold currently trains: histgb, catboost, tabpfn (if installed).
It caches fold-level predictions + metrics under artifacts/.

In [ ]:
import json, os
import yaml
from sklearn.pipeline import Pipeline

from src.data import DataSpec, load_dataframe, build_feature_sets
from src.splits import load_splits
from src.metrics import compute_binary_metrics, to_dict
from src.thresholding import choose_thresholds
from src.calibration import Calibrator
from src.models.wrappers_sklearn import build_preprocessor, make_histgb_classifier, make_catboost_classifier

cfg = yaml.safe_load(open("configs/base.yaml"))
b_cfg = yaml.safe_load(open("configs/baselines.yaml"))

spec = DataSpec(
    csv_path=cfg["data"]["csv_path"],
    id_col=cfg["data"]["id_col"],
    target_col=cfg["data"]["target_col"],
    sbp_2classes_col=cfg["data"]["sbp_2classes_col"],
    positive_class_value=cfg["data"]["positive_class_value"],
    sbp_col=cfg["data"]["sbp_col"],
    sbp_threshold=cfg["data"]["sbp_threshold"],
    leakage_cols=tuple(cfg["data"]["leakage_cols"]),
    demographic_cols=tuple(cfg["data"]["demographic_cols"]),
    categorical_cols=tuple(cfg["data"]["categorical_cols"]),
)

df = load_dataframe(spec.csv_path)
splits = load_splits("artifacts/splits/outer_splits.json")

os.makedirs("artifacts/preds", exist_ok=True)
os.makedirs("artifacts/metrics", exist_ok=True)
os.makedirs("artifacts/calib", exist_ok=True)

class DenseTransformer:
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.toarray() if hasattr(X, "toarray") else X

for fs_name, fs_cfg in cfg["data"]["feature_sets"].items():
    X, y, groups, info, cat_cols = build_feature_sets(df, spec, include_demographics=fs_cfg["include_demographics"])

    for model_item in b_cfg["models"]:
        name = model_item["name"]
        if name not in ("histgb", "catboost", "tabpfn"):
            print("Skipping for now:", name)
            continue

        for fold, sp in enumerate(splits):
            tr_idx, te_idx = sp["train_idx"], sp["test_idx"]
            Xtr, Xte = X.iloc[tr_idx], X.iloc[te_idx]
            ytr, yte = y.iloc[tr_idx].values, y.iloc[te_idx].values

            pre = build_preprocessor(Xtr, cat_cols=cat_cols)

            if name == "histgb":
                clf = make_histgb_classifier(model_item["params"])
                pipe = Pipeline([("pre", pre), ("clf", clf)])
                pipe.fit(Xtr, ytr)
                p_tr = pipe.predict_proba(Xtr)[:, 1]
                p_te = pipe.predict_proba(Xte)[:, 1]

            elif name == "catboost":
                clf = make_catboost_classifier(model_item["params"])
                pipe = Pipeline([("pre", pre), ("clf", clf)])
                pipe.fit(Xtr, ytr)
                p_tr = pipe.predict_proba(Xtr)[:, 1]
                p_te = pipe.predict_proba(Xte)[:, 1]

            elif name == "tabpfn":
                try:
                    from tabpfn import TabPFNClassifier
                except Exception as e:
                    print("TabPFN not installed:", e)
                    continue
                clf = TabPFNClassifier()
                pipe = Pipeline([("pre", pre), ("dense", DenseTransformer()), ("clf", clf)])
                pipe.fit(Xtr, ytr)
                p_tr = pipe.predict_proba(Xtr)[:, 1]
                p_te = pipe.predict_proba(Xte)[:, 1]

            if cfg["calibration"]["enabled"]:
                calib = Calibrator(method=cfg["calibration"]["method"]).fit(ytr, p_tr)
                p_tr_c = calib.transform(p_tr)
                p_te_c = calib.transform(p_te)
            else:
                p_tr_c, p_te_c = p_tr, p_te

            thr = choose_thresholds(ytr, p_tr_c, min_recall=cfg["screening"]["min_recall"])
            m_f1 = compute_binary_metrics(yte, p_te_c, threshold=thr.threshold_f1)
            m_sc = compute_binary_metrics(yte, p_te_c, threshold=thr.threshold_screening)

            json.dump(
                {
                    "fs": fs_name,
                    "model": name,
                    "fold": fold,
                    "y_true": yte.tolist(),
                    "p_raw": p_te.tolist(),
                    "p_cal": p_te_c.tolist(),
                    "threshold_f1": thr.threshold_f1,
                    "threshold_screen": thr.threshold_screening,
                },
                open(f"artifacts/preds/{fs_name}__{name}__fold{fold}.json", "w"),
                indent=2,
            )

            json.dump(
                {
                    "fs": fs_name,
                    "model": name,
                    "fold": fold,
                    "f1_opt": to_dict(m_f1),
                    "screening": to_dict(m_sc),
                    "thresholds": {"f1": thr.threshold_f1, "screening": thr.threshold_screening},
                },
                open(f"artifacts/metrics/{fs_name}__{name}__fold{fold}.json", "w"),
                indent=2,
            )

            print("[OK]", fs_name, name, "fold", fold)